# Slang Generation Demo

This demo will illustrate the basic workflow of the slang generation code accompanying the TACL paper *[A Computational Framework for Slang Generation](https://direct.mit.edu/tacl/article/doi/10.1162/tacl_a_00378/100687/A-Computational-Framework-for-Slang-Generation)* using slang definition data from Urban Dictionary (UD) and conventional definition data from WordNet.

To run this tutorial, you will need the following dependencies:

- Python 3
- Numpy
- Scipy
- tqdm
- NLTK
- Gensim
- PyTorch (torch)
- SBERT (sentence_transformers)
- [CatGO](https://github.com/zhewei-sun/CatGO)


In [ ]:
import numpy as np
import torch
import shutil

We first create a symbolic link pointing to the library. You will need to change the destination if your code sits in a different directory. 

In [ ]:
! ln -s ../Code slanggen

CatGO is a library that optimizes and runs models of categorization and can be obtained [here](https://github.com/zhewei-sun/CatGO). Once you have downloaded the code, please link it by replacing the target directory of the simlink below. 

In [ ]:
! ln -s ../../CatGO CatGO
import nltk
nltk.download('stopwords')

In [ ]:
from slanggen.util import *
from slanggen.dataloader import WN_Dataset, Urban_Dataset, OSD_Dataset, ZH_Dataset
from slanggen.encoder import FTEncoder, FTCachedEncoder, SBertEncoder, SenseEncoder, dump_vanilla_embeddings
from slanggen.contrastive import SlangGenTrainer
from slanggen.model import SlangGenModel

Specify a PyTorch device if necessary:

In [ ]:
torch.cuda.set_device(0)

Load conventional definition data using the builtin dataloaders. The *.npy* file loaded below contains a pre-processed version of WordNet definition sentences for all words that appear in both WordNet and UD.

In [ ]:
wn_data = WN_Dataset('OD_OSD.npy')
print(wn_data)

zh_conv_data = WN_Dataset('ZH_zh_conv_data_chime.npy')
print(zh_conv_data)

ru_conv_data = WN_Dataset('RU_ru_conv_data.npy')
print(ru_conv_data)

Load slang definition data. The *.npy* file loaded below is a pre-processed version of the data released in this repository.

In [ ]:
# English slang data
OSD_wn_data = OSD_Dataset('OSD_V2.npy', wn_data)
print(OSD_wn_data)

# Chinese slang data (in Chinese)
zh_data = ZH_Dataset('ZH_zh_slang_data_chime.npy', zh_conv_data)
print(zh_data)

# Russian slang data (in Russian)
ru_data = ZH_Dataset('RU_ru_slang_data.npy', ru_conv_data)
print(ru_data)

If you wish to use your own dataset, please create a dataloader object inheriting either *ConvDataset* or *SlangDataset* abstract classes found in *dataloader.py* and following the example data specifications in *dataloader.WN_Dataset* and *dataloader.Urban_Dataset*.

Now let's create a directory to store our results and load in some pre-generated data indices for train-test split:

In [ ]:
shutil.rmtree('Results')  
create_directory('Results')

In [ ]:
out_dir='Results/'

dataset_osd = OSD_wn_data
slang_inds_osd = DataIndex(np.load('train_ind_OD_OSD.npy'), np.load('dev_ind_OD_OSD.npy'), np.load('test_ind_OD_OSD.npy'))

# Here, zh_zh means the ZH dataset written in ZH. train_ind_zh means the ZH training index in English
dataset_zh_zh = zh_data
slang_inds_zh_zh = DataIndex(np.load('train_ind_ZH_zh_chime.npy'), np.load('dev_ind_ZH_zh_chime.npy'), np.load('test_ind_ZH_zh_chime.npy'))

dataset_ru_ru = ru_data
slang_inds_ru_ru = DataIndex(np.load('train_ind_ru_ru.npy'), np.load('dev_ind_ru_ru.npy'), np.load('test_ind_ru_ru.npy'))

The following encoder objects initializes a fastText encoder used for collaborative filtering. *FTEncoder* can be used to read in the original fastText embedding file. For efficiency, we have cached the words we need and use a cached encoder instead.

ft_encoder = FTEncoder('path to crawl-300d-2M-subword.vec')

In [ ]:
ft_encoder = FTCachedEncoder('ft_embed_cache_Urban.pickle')

The following commands sets up the contrastive trainer and the slang generation model:

In [ ]:
trainer_osd = SlangGenTrainer(dataset_osd, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)

trainer_zh_zh = SlangGenTrainer(dataset_zh_zh, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)

trainer_ru_ru = SlangGenTrainer(dataset_ru_ru, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)

You can modify the 'embed_name' param below to choose the sense encoding model, here is a list of supported models:
- bert-base-nli-mean-tokens -> 'SBERT_contrastive' (default)
- sentence-t5-base -> 'SBERT_t5'
- paraphrase-multilingual-MiniLM-L12-v2 -> 'SBERT-multilingual-MiniLM-L12-v2'
- LaBSE -> 'SBERT_LaBSE'
- paraphrase-multilingual-mpnet-base-v2 -> 'SBERT_mpnet'
- intfloat/multilingual-e5-base -> 'SBERT_e5_base'
- intfloat/multilingual-e5-large -> 'SBERT_e5_large'

In [ ]:
# model_osd = SlangGenModel(trainer_osd, data_dir=out_dir, embed_name='SBERT_mpnet')

model_zh_zh = SlangGenModel(trainer_zh_zh, data_dir=out_dir, embed_name='SBERT_mpnet')

# model_ru_ru = SlangGenModel(trainer_ru_ru, data_dir=out_dir, embed_name='SBERT_mpnet')

params = {'embed_name':'SBERT_mpnet', 'out_name':'predictions', 'model':'cf_prototype_5', 'prior':None, 'prior_name':'uniform', 'contr_params':None}


Invoke *model.train_contrastive* to train the contrastively learned sense embedding model:

Note: you can set mode to 'head' to train only the triplet head, or 'whole' to train both the sense encoding and the head. Also it's optional to change the fold_name if using a new dataset.

In [ ]:
# model_osd.train_contrastive(slang_inds_osd, fold_name='osd_wn', params=params, mode='whole')

model_zh_zh.train_contrastive(slang_inds_zh_zh, fold_name='zh_zh', params=params, mode='head')

# model_ru_ru.train_contrastive(slang_inds_ru_ru, fold_name='ru_ru', params=params, mode='head')


Once the contrastive represention has been trained. We run categorization models to perform few-shot learning:

In [ ]:
# Test how the model perform on different languages
params['embed_name'] = 'SBERT_mpnet' 

!mkdir -p Results/test_en/SBERT_data
!mkdir -p Results/test_zh/SBERT_data
!mkdir -p Results/test_ru/SBERT_data

!cp Results/zh_zh/SBERT_data/SBERT_mpnet_with_head.pt Results/test_en/SBERT_data/ # with_head.pt or whole_finetuned.pt
# !cp Results/ru_ru/SBERT_data/SBERT_mpnet_whole_finetuned.pt Results/test_en/SBERT_data/

trainer_test_en = SlangGenTrainer(OSD_wn_data, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_test_en   = SlangGenModel(trainer_test_en, data_dir=out_dir, embed_name='SBERT_mpnet')

trainer_test_en.get_trained_embeddings(slang_inds_osd, fold_name='test_en', model_path='SBERT_mpnet')

model_test_en.train_categorization(slang_inds_osd, fold_name='test_en', params=params)
results_en = model_test_en.get_results(fold_name='test_en', mode='train', params=params)

print('Performance on train split - EN')
N_train_dev_en = OSD_wn_data.N_total - slang_inds_osd.test.shape[0]
train_rankings_en = get_rankings(results_en, np.arange(N_train_dev_en), OSD_wn_data.vocab_ids[np.concatenate(slang_inds_osd)])
np.mean(get_roc(train_rankings_en, OSD_wn_data.V))

In [ ]:
model_test_en.predict_testset(slang_inds_osd, fold_name='test_en', params=params)
results_en = model_test_en.get_results(fold_name='test_en', mode='test', params=params)

print('Performance on test split - EN')
inds = slang_inds_osd.test                     
labels = OSD_wn_data.vocab_ids                 
test_rankings_en = get_rankings(results_en, inds, labels)
np.mean(get_roc(test_rankings_en, OSD_wn_data.V))

In [ ]:
!cp Results/zh_zh/SBERT_data/SBERT_mpnet_with_head.pt Results/test_zh/SBERT_data/
# !cp Results/ru_ru/SBERT_data/SBERT_mpnet_whole_finetuned.pt Results/test_zh/SBERT_data/

trainer_test_zh = SlangGenTrainer(zh_data, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_test_zh   = SlangGenModel(trainer_test_zh, data_dir=out_dir, embed_name='SBERT_mpnet')

trainer_test_zh.get_trained_embeddings(slang_inds_zh_zh, fold_name='test_zh', model_path='SBERT_mpnet')

model_test_zh.train_categorization(slang_inds_zh_zh, fold_name='test_zh', params=params)
results_zh = model_test_zh.get_results(fold_name='test_zh', mode='train', params=params)

print('Performance on train split - ZH')
N_train_dev_zh = zh_data.N_total - slang_inds_zh_zh.test.shape[0]
train_rankings_zh = get_rankings(results_zh, np.arange(N_train_dev_zh), zh_data.vocab_ids[np.concatenate(slang_inds_zh_zh)])
np.mean(get_roc(train_rankings_zh, zh_data.V))

In [ ]:
model_test_zh.predict_testset(slang_inds_zh_zh, fold_name='test_zh', params=params)
results_zh = model_test_zh.get_results(fold_name='test_zh', mode='test', params=params)

print('Performance on test split - ZH')
inds = slang_inds_zh_zh.test                      
labels = zh_data.vocab_ids                 
test_rankings_zh = get_rankings(results_zh, inds, labels)
np.mean(get_roc(test_rankings_zh, zh_data.V))

In [ ]:
!cp Results/zh_zh/SBERT_data/SBERT_mpnet_with_head.pt Results/test_ru/SBERT_data/
# !cp Results/ru_ru/SBERT_data/SBERT_mpnet_whole_finetuned.pt Results/test_ru/SBERT_data/

trainer_test_ru = SlangGenTrainer(ru_data, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_test_ru   = SlangGenModel(trainer_test_ru, data_dir=out_dir, embed_name='SBERT_mpnet')

trainer_test_ru.get_trained_embeddings(slang_inds_ru_ru, fold_name='test_ru', model_path='SBERT_mpnet')

model_test_ru.train_categorization(slang_inds_ru_ru, fold_name='test_ru', params=params)
results_ru = model_test_ru.get_results(fold_name='test_ru', mode='train', params=params)

print('Performance on train split - RU')
N_train_dev_ru = ru_data.N_total - slang_inds_ru_ru.test.shape[0]
train_rankings_ru = get_rankings(results_ru, np.arange(N_train_dev_ru), ru_data.vocab_ids[np.concatenate(slang_inds_ru_ru)])
np.mean(get_roc(train_rankings_ru, ru_data.V))

In [ ]:
model_test_ru.predict_testset(slang_inds_ru_ru, fold_name='test_ru', params=params)
results_ru = model_test_ru.get_results(fold_name='test_ru', mode='test', params=params)

print('Performance on test split - RU')
inds = slang_inds_ru_ru.test                      
labels = ru_data.vocab_ids                 
test_rankings_ru = get_rankings(results_ru, inds, labels)
np.mean(get_roc(test_rankings_ru, ru_data.V))

In [ ]:
# We add a baseline here: use a vanilla mpnet model, skipping train_contrastive, to run train_categorization directly

trainer_test_en = SlangGenTrainer(OSD_wn_data, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
trainer_test_zh = SlangGenTrainer(zh_data, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
trainer_test_ru = SlangGenTrainer(ru_data, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)

dump_vanilla_embeddings(trainer_test_en, slang_inds_osd, fold_name='en_vanilla', embed_name=params['embed_name'])
dump_vanilla_embeddings(trainer_test_zh, slang_inds_zh_zh, fold_name='zh_vanilla', embed_name=params['embed_name'])
dump_vanilla_embeddings(trainer_test_ru, slang_inds_ru_ru, fold_name='ru_vanilla', embed_name=params['embed_name'])

model_v_en = SlangGenModel(trainer_test_en, data_dir=out_dir, embed_name=params['embed_name'])
model_v_zh = SlangGenModel(trainer_test_zh, data_dir=out_dir, embed_name=params['embed_name'])
model_v_ru = SlangGenModel(trainer_test_ru, data_dir=out_dir, embed_name=params['embed_name'])

ds_en = trainer_test_en.dataset
ds_zh = trainer_test_zh.dataset
ds_ru = trainer_test_ru.dataset

# EN
model_v_en.train_categorization(slang_inds_osd, fold_name='en_vanilla', params=params)
res_en_tr = model_v_en.get_results(fold_name='en_vanilla', mode='train', params=params)
model_v_en.predict_testset(slang_inds_osd, fold_name='en_vanilla', params=params)
res_en_te = model_v_en.get_results(fold_name='en_vanilla', mode='test', params=params)

# ZH
model_v_zh.train_categorization(slang_inds_zh_zh, fold_name='zh_vanilla', params=params)
res_zh_tr = model_v_zh.get_results(fold_name='zh_vanilla', mode='train', params=params)
model_v_zh.predict_testset(slang_inds_zh_zh, fold_name='zh_vanilla', params=params)
res_zh_te = model_v_zh.get_results(fold_name='zh_vanilla', mode='test', params=params)

# RU
model_v_ru.train_categorization(slang_inds_ru_ru, fold_name='ru_vanilla', params=params)
res_ru_tr = model_v_ru.get_results(fold_name='ru_vanilla', mode='train', params=params)
model_v_ru.predict_testset(slang_inds_ru_ru, fold_name='ru_vanilla', params=params)
res_ru_te = model_v_ru.get_results(fold_name='ru_vanilla', mode='test', params=params)

def _all_inds(si):
    return np.concatenate((si.train, si.dev, si.test))

def auc_from_results(results, ds, slang_inds, split='test'):
    if split == 'train':
        inds = np.concatenate([slang_inds.train, slang_inds.dev])  
    else:
        inds = slang_inds.test                                     
    labels = ds.vocab_ids                                          
    rankings = get_rankings(results, inds, labels)                 
    return float(np.mean(get_roc(rankings, ds.V)))

print('EN vanilla AUC (train):', auc_from_results(res_en_tr, ds_en, slang_inds_osd, 'train'))
print('EN vanilla AUC (test): ', auc_from_results(res_en_te, ds_en, slang_inds_osd, 'test'))

print('ZH vanilla AUC (train):', auc_from_results(res_zh_tr, ds_zh, slang_inds_zh_zh, 'train'))
print('ZH vanilla AUC (test): ', auc_from_results(res_zh_te, ds_zh, slang_inds_zh_zh, 'test'))

print('RU vanilla AUC (train):', auc_from_results(res_ru_tr, ds_ru, slang_inds_ru_ru, 'train'))
print('RU vanilla AUC (test): ', auc_from_results(res_ru_te, ds_ru, slang_inds_ru_ru, 'test'))
